# Pipeline final - XGBoost

#### Pipeline final do Projeto de acordo com resultado do Plano de Experimentação

In [16]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import RepeatedKFold, cross_val_score
from scipy.stats import zscore

#### Função auxiliar de remoção de outliers

In [17]:
def remover_outliers(df, colunas, threshold=3):
    z_scores = np.abs(zscore(df[colunas]))
    mask = (z_scores < threshold).all(axis=1)
    return df[mask]

#### Leitura do dataframe

In [18]:
df = pd.read_csv('../data/dataframes/pre-processado/por-municipio/completo/dados-manaus-preprocessado.csv')

#### Pipeline

In [21]:
df.columns = df.columns.str.strip().str.lower()

X = df.drop(columns=['vazao', 'municipio', 'uf'])
y = df['vazao']

df_completo = X.copy()
df_completo['target'] = y

num_cols = df_completo.select_dtypes(include=['int64', 'float64']).columns.tolist()
num_cols = [col for col in num_cols if col != 'target']

print(f"Antes da remoção de outliers: {df_completo.shape[0]} linhas")
df_sem_outliers = remover_outliers(df_completo, num_cols)
print(f"Depois da remoção de outliers: {df_sem_outliers.shape[0]} linhas")

X_clean = df_sem_outliers.drop(columns='target')
y_clean = df_sem_outliers['target']

num_cols = X_clean.select_dtypes(include=['int64', 'float64']).columns.tolist()

preprocessador = ColumnTransformer([
    ('scale', MinMaxScaler(), num_cols)
], remainder='passthrough')

pipeline = Pipeline([
    ('prep', preprocessador),
    ('xgb', XGBRegressor(learning_rate=0.3, max_depth=5, random_state=42, n_jobs=-1))
])

cv = RepeatedKFold(n_splits=5, n_repeats=2, random_state=42)

r2_scores = cross_val_score(pipeline, X_clean, y_clean, cv=cv, scoring='r2')
print(f"\nR²: {r2_scores.mean():.4f}")


Antes da remoção de outliers: 12022 linhas
Depois da remoção de outliers: 10903 linhas

R²: 0.9871
